In [1]:
# ============================================
# QUESTION 2: Data Loading and Augmentation Using Keras
# COMPLETE SELF-CONTAINED COLAB NOTEBOOK
# ============================================

import os
import numpy as np
import random
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

print("="*60)
print("QUESTION 2: Data Loading and Augmentation Using Keras")
print("="*60)

# ============================================
# STEP 1: Create sample dataset
# ============================================

print("\n📁 Creating sample dataset...")

def create_sample_dataset():
    os.makedirs('./images_dataSAT/class_0_non_agri/', exist_ok=True)
    os.makedirs('./images_dataSAT/class_1_agri/', exist_ok=True)

    # Create 20 non-agri images
    for i in range(20):
        img = np.random.randint(0, 255, (84, 80, 3), dtype=np.uint8)
        if i % 2 == 0:
            img[20:60, 30:50] = [200, 200, 200]
        cv2.imwrite(f'./images_dataSAT/class_0_non_agri/non_agri_{i:03d}.png', img)

    # Create 25 agri images
    for i in range(25):
        img = np.random.randint(0, 255, (84, 80, 3), dtype=np.uint8)
        for _ in range(5):
            x = random.randint(0, 70)
            y = random.randint(0, 74)
            img[y:y+10, x:x+10] = [50, 200, 50]
        cv2.imwrite(f'./images_dataSAT/class_1_agri/agri_{i:03d}.png', img)

    print("✅ Sample dataset created!")

if not os.path.exists('./images_dataSAT'):
    create_sample_dataset()
else:
    print("✅ Dataset already exists!")

# ============================================
# Task 1: Create all_image_paths
# ============================================

print("\n" + "="*50)
print("Task 1: Create all_image_paths")
print("="*50)

class_0_dir = './images_dataSAT/class_0_non_agri/'
class_1_dir = './images_dataSAT/class_1_agri/'

class_0_paths = [os.path.join(class_0_dir, f) for f in os.listdir(class_0_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
class_1_paths = [os.path.join(class_1_dir, f) for f in os.listdir(class_1_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
all_image_paths = class_0_paths + class_1_paths

print(f"Total image paths: {len(all_image_paths)}")
print(f"Class 0 (non-agri): {len(class_0_paths)}")
print(f"Class 1 (agri): {len(class_1_paths)}")

# ============================================
# Task 2: Create temp list binding paths and labels
# ============================================

print("\n" + "="*50)
print("Task 2: Create temp list with paths and labels")
print("="*50)

labels_0 = [0] * len(class_0_paths)
labels_1 = [1] * len(class_1_paths)
all_labels = labels_0 + labels_1
temp = list(zip(all_image_paths, all_labels))
random.shuffle(temp)

print("5 random samples:")
for sample in temp[:5]:
    print(f"  {sample}")

# ============================================
# Task 3: Custom data generator (batch size = 8)
# ============================================

print("\n" + "="*50)
print("Task 3: Custom data generator (batch_size=8)")
print("="*50)

def custom_data_generator(image_paths, labels, batch_size=8):
    while True:
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            batch_images = []
            for path in batch_paths:
                img = tf.keras.preprocessing.image.load_img(path, target_size=(80, 84))
                img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
                batch_images.append(img_array)
            yield np.array(batch_images), np.array(batch_labels)

generator = custom_data_generator(all_image_paths, all_labels, batch_size=8)
batch = next(generator)

print(f"Batch images shape: {batch[0].shape}")
print(f"Batch labels shape: {batch[1].shape}")

# ============================================
# Task 4: Create validation data (batch size = 8)
# ============================================

print("\n" + "="*50)
print("Task 4: Create validation data (batch_size=8)")
print("="*50)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

val_generator = val_datagen.flow_from_directory(
    './images_dataSAT/',
    target_size=(80, 84),
    batch_size=8,
    class_mode='binary',
    subset='validation',
    shuffle=True
)

print(f"\n✅ Validation generator created successfully!")
print(f"Validation samples: {val_generator.samples}")

# ============================================
# Summary
# ============================================

print("\n" + "="*50)
print("✅ QUESTION 2 COMPLETED SUCCESSFULLY!")
print("="*50)
print(f"\n📊 Summary:")
print(f"  - Total images: {len(all_image_paths)}")
print(f"  - Non-agri images: {len(class_0_paths)}")
print(f"  - Agri images: {len(class_1_paths)}")
print(f"  - Validation samples: {val_generator.samples}")
print(f"  - Batch size: 8")

QUESTION 2: Data Loading and Augmentation Using Keras

📁 Creating sample dataset...
✅ Sample dataset created!

Task 1: Create all_image_paths
Total image paths: 45
Class 0 (non-agri): 20
Class 1 (agri): 25

Task 2: Create temp list with paths and labels
5 random samples:
  ('./images_dataSAT/class_0_non_agri/non_agri_015.png', 0)
  ('./images_dataSAT/class_1_agri/agri_016.png', 1)
  ('./images_dataSAT/class_0_non_agri/non_agri_013.png', 0)
  ('./images_dataSAT/class_0_non_agri/non_agri_018.png', 0)
  ('./images_dataSAT/class_1_agri/agri_020.png', 1)

Task 3: Custom data generator (batch_size=8)
Batch images shape: (8, 80, 84, 3)
Batch labels shape: (8,)

Task 4: Create validation data (batch_size=8)
Found 9 images belonging to 2 classes.

✅ Validation generator created successfully!
Validation samples: 9

✅ QUESTION 2 COMPLETED SUCCESSFULLY!

📊 Summary:
  - Total images: 45
  - Non-agri images: 20
  - Agri images: 25
  - Validation samples: 9
  - Batch size: 8
